In [ ]:
# -*- coding: utf-8 -*-
"""
Enhanced DGRNN v2: Dispatch-Aware Residual Gated Neural Network
================================================================

Goal
----
Improve CPU-usage prediction while preserving a strict leakage-free design.

Main methodological upgrade
----------------------------
1. Dispatch-only features are used as model inputs.
2. Random Forest remains the strong nonlinear baseline.
3. A 3-fold out-of-fold Random Forest prediction is generated on TRAIN only.
4. The proposed neural network learns the RF residual rather than relearning
   the whole target from scratch.
5. The neural network uses gated residual blocks + feature-cross block +
   LayerNorm + GELU + AdamW.
6. Regression and high-CPU classification are trained jointly as an auxiliary
   multi-task objective. The classification threshold is derived from TRAIN only.
7. Tail-aware regression weighting emphasizes the upper CPU tail.
8. Early stopping restores the best validation checkpoint.
9. A validation-only blend between RF and the residual DGRNN is searched.
10. TEST is never used for architecture, hyperparameter, threshold, or blend
    selection. It is used only for final reporting.

Important scientific note
-------------------------
Call this a "proposed Dispatch-Aware Residual Gated Neural Network (DGRNN-v2)"
in the paper. Do not claim global architectural novelty without a literature
review establishing that claim.
"""

# ============================================================
# 0. Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

import os
import ast
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# 1. Configuration
# ============================================================
RANDOM_STATE = 42

DRIVE_DIR = "/content/drive/MyDrive/AI-GoogleCluster"
RESULTS_DIR = os.path.join(DRIVE_DIR, "results_enhanced_dgrnn_v2")
TABLE_DIR = os.path.join(RESULTS_DIR, "tables")
FIGURE_DIR = os.path.join(RESULTS_DIR, "figures")
MODEL_DIR = os.path.join(RESULTS_DIR, "models")

for d in [DRIVE_DIR, RESULTS_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

DATASET_NAME = "derrickmwiti/google-2019-cluster-sample"
CACHE_CSV_PATH = os.path.join(DRIVE_DIR, "borg_traces_data.csv")

# Runtime controls
OOF_FOLDS = 3
RF_TREES = 300
ET_TREES = 300
EPOCHS = 120
BATCH_SIZE = 512
PATIENCE = 12
LEARNING_RATE = 6e-4

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ============================================================
# 2. Install / imports
# ============================================================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "tensorflow"], check=False)

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

tf.keras.utils.set_random_seed(RANDOM_STATE)

print("TensorFlow:", tf.__version__)
print("Results:", RESULTS_DIR)

# ============================================================
# 3. Load data
# ============================================================
def load_data(cache_csv_path=CACHE_CSV_PATH):
    if os.path.exists(cache_csv_path):
        df = pd.read_csv(cache_csv_path, low_memory=False)
        print(f"Loaded cached dataset: {cache_csv_path}")
        print("Shape:", df.shape)
        return df

    import kagglehub
    path = kagglehub.dataset_download(DATASET_NAME)
    src_csv = os.path.join(path, "borg_traces_data.csv")
    df = pd.read_csv(src_csv, low_memory=False)
    df.to_csv(cache_csv_path, index=False)
    print("Downloaded:", src_csv)
    print("Cached:", cache_csv_path)
    print("Shape:", df.shape)
    return df


df_raw = load_data()
print("\nColumns:")
print(df_raw.columns.tolist())

# ============================================================
# 4. Leakage-safe feature policy
# ============================================================
BASE_FEATURES = [
    "scheduling_class",
    "priority",
    "resource_request_cpus",
    "resource_request_memory",
    "vertical_scaling",
    "scheduler",
    "collection_type",
]

TARGET = "realized_cpu_usage"

POST_EXECUTION_FIELDS = [
    "start_time", "end_time", "assigned_memory", "page_cache_memory",
    "cycles_per_instruction", "memory_accesses_per_instruction",
    "average_usage", "maximum_usage", "random_sample_usage",
    "cpu_usage_distribution", "tail_cpu_usage_distribution", "failed",
]

# ============================================================
# 5. Robust parser for nested resource/usage dictionaries
# ============================================================
def parse_nested_value(value, key):
    if value is None:
        return np.nan
    if isinstance(value, float) and np.isnan(value):
        return np.nan

    if isinstance(value, dict):
        try:
            return float(value.get(key, np.nan))
        except Exception:
            return np.nan

    if isinstance(value, str):
        s = value.strip()
        if not s:
            return np.nan

        for parser in (ast.literal_eval, json.loads):
            try:
                obj = parser(s)
                if isinstance(obj, dict):
                    return float(obj.get(key, np.nan))
            except Exception:
                pass

    try:
        return float(value)
    except Exception:
        return np.nan


# ============================================================
# 6. Build target + dispatch-time features
# ============================================================
def build_feature_target_frame(df):
    required_raw = [
        "scheduling_class", "priority", "resource_request",
        "vertical_scaling", "scheduler", "collection_type", "average_usage",
    ]

    missing = [c for c in required_raw if c not in df.columns]
    if missing:
        raise KeyError("Missing required raw columns: " + ", ".join(missing))

    sub = df[required_raw].copy()

    sub["resource_request_cpus"] = sub["resource_request"].apply(
        lambda x: parse_nested_value(x, "cpus")
    )
    sub["resource_request_memory"] = sub["resource_request"].apply(
        lambda x: parse_nested_value(x, "memory")
    )
    sub[TARGET] = sub["average_usage"].apply(
        lambda x: parse_nested_value(x, "cpus")
    )

    for c in BASE_FEATURES + [TARGET]:
        sub[c] = pd.to_numeric(sub[c], errors="coerce")

    sub = sub.replace([np.inf, -np.inf], np.nan)

    print("\nRaw rows:", f"{len(df):,}")
    print("Missing parsed values:")
    print(sub[["resource_request_cpus", "resource_request_memory", TARGET]].isna().sum())

    before = len(sub)
    sub = sub.dropna(subset=BASE_FEATURES + [TARGET]).copy()
    sub = sub[sub[TARGET] >= 0].copy()

    print("Rows before preprocessing:", f"{before:,}")
    print("Rows after preprocessing: ", f"{len(sub):,}")
    print("Rows removed:             ", f"{before-len(sub):,}")

    return sub[BASE_FEATURES + [TARGET]].copy()


data = build_feature_target_frame(df_raw)
X_base = data[BASE_FEATURES].copy()
y = data[TARGET].astype(float).copy()

# ============================================================
# 7. Dispatch-aware feature engineering
# ============================================================
def engineer_dispatch_features(X):
    out = X.copy()
    eps = 1e-8

    cpu = np.clip(out["resource_request_cpus"].values, 0, None)
    mem = np.clip(out["resource_request_memory"].values, 0, None)
    priority = out["priority"].values
    sched_class = out["scheduling_class"].values
    collection = out["collection_type"].values
    scheduler = out["scheduler"].values
    vertical = out["vertical_scaling"].values

    out["log_request_cpus"] = np.log1p(cpu)
    out["log_request_memory"] = np.log1p(mem)
    out["sqrt_request_cpus"] = np.sqrt(cpu)
    out["sqrt_request_memory"] = np.sqrt(mem)

    mem_per_cpu = mem / (cpu + eps)
    out["request_memory_per_cpu"] = mem_per_cpu
    out["log_memory_per_cpu"] = np.log1p(np.clip(mem_per_cpu, 0, None))

    # Resource-policy interactions
    out["priority_x_cpu"] = priority * cpu
    out["priority_x_log_cpu"] = priority * np.log1p(cpu)
    out["priority_x_memory"] = priority * np.log1p(mem)
    out["class_x_cpu"] = sched_class * cpu
    out["class_x_memory"] = sched_class * np.log1p(mem)
    out["collection_x_cpu"] = collection * cpu
    out["scheduler_x_cpu"] = scheduler * cpu
    out["vertical_x_cpu"] = vertical * cpu

    # Nonlinear resource terms
    out["cpu_squared"] = cpu ** 2
    out["log_cpu_squared"] = np.log1p(cpu) ** 2
    out["log_memory_squared"] = np.log1p(mem) ** 2
    out["cpu_memory_log_product"] = np.log1p(cpu) * np.log1p(mem)
    out["cpu_over_memory"] = cpu / (np.log1p(mem) + 1.0)

    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out.astype(np.float32)


X = engineer_dispatch_features(X_base)
print("\nEngineered feature count:", X.shape[1])
print(X.columns.tolist())

# ============================================================
# 8. Exact 70/15/15 split
# ============================================================
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

print("\n" + "=" * 75)
print("DATASET REPORT -- V2")
print("=" * 75)
print(f"Raw rows:                 {len(df_raw):,}")
print(f"Rows after preprocessing: {len(X):,}")
print(f"Train set size:           {len(X_train):,} ({len(X_train)/len(X):.1%})")
print(f"Validation set size:      {len(X_val):,} ({len(X_val)/len(X):.1%})")
print(f"Test set size:            {len(X_test):,} ({len(X_test)/len(X):.1%})")
print("=" * 75)

# ============================================================
# 9. Scaling -- TRAIN ONLY
# ============================================================
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

# ============================================================
# 10. Metrics
# ============================================================
def regression_metrics(y_true, y_pred):
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def print_regression_result(name, y_true, pred):
    m = regression_metrics(y_true, pred)
    print(
        f"[{name}] MAE={m['MAE']:.6f}  "
        f"RMSE={m['RMSE']:.6f}  R2={m['R2']:.6f}"
    )
    return m

# ============================================================
# 11. Strong tree baselines
# ============================================================
regression_results = []
predictions = {}

rf = RandomForestRegressor(
    n_estimators=RF_TREES,
    min_samples_leaf=2,
    max_features=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_val = np.maximum(rf.predict(X_val), 0)
rf_test = np.maximum(rf.predict(X_test), 0)

m = print_regression_result("Random Forest", y_test, rf_test)
regression_results.append({"Model": "Random Forest", **m})
predictions["Random Forest"] = rf_test

et = ExtraTreesRegressor(
    n_estimators=ET_TREES,
    min_samples_leaf=2,
    max_features=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
et.fit(X_train, y_train)
et_val = np.maximum(et.predict(X_val), 0)
et_test = np.maximum(et.predict(X_test), 0)

m = print_regression_result("Extra Trees", y_test, et_test)
regression_results.append({"Model": "Extra Trees", **m})
predictions["Extra Trees"] = et_test

# ============================================================
# 12. High-CPU threshold from TRAIN ONLY
# ============================================================
high_cpu_threshold = float(y_train.quantile(0.75))
y_train_c = (y_train >= high_cpu_threshold).astype(np.float32)
y_val_c = (y_val >= high_cpu_threshold).astype(np.float32)
y_test_c = (y_test >= high_cpu_threshold).astype(np.float32)

print("\nHigh-CPU threshold (TRAIN 75%):", f"{high_cpu_threshold:.6f}")
print("Train class distribution:", y_train_c.value_counts().sort_index().to_dict())
print("Val class distribution:  ", y_val_c.value_counts().sort_index().to_dict())
print("Test class distribution: ", y_test_c.value_counts().sort_index().to_dict())

# ============================================================
# 13. OOF Random Forest predictions on TRAIN
# ============================================================
# This is the key leakage-control step for residual learning.
# Each training row receives a prediction from a forest that did NOT train on it.
# ============================================================
print("\nGenerating 3-fold OOF Random Forest predictions for residual learning...")

oof_rf = np.zeros(len(X_train), dtype=np.float32)
kf = KFold(n_splits=OOF_FOLDS, shuffle=True, random_state=RANDOM_STATE)

X_train_reset = X_train.reset_index(drop=True)
y_train_reset = y_train.reset_index(drop=True)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_reset), start=1):
    fold_rf = RandomForestRegressor(
        n_estimators=180,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
    )
    fold_rf.fit(X_train_reset.iloc[tr_idx], y_train_reset.iloc[tr_idx])
    oof_rf[va_idx] = np.maximum(
        fold_rf.predict(X_train_reset.iloc[va_idx]), 0
    ).astype(np.float32)
    print(f"  Fold {fold}/{OOF_FOLDS} complete")

residual_train = y_train_reset.values.astype(np.float32) - oof_rf
print("OOF residual mean:", float(residual_train.mean()))
print("OOF residual std :", float(residual_train.std()))

# ============================================================
# 14. Proposed V2 architecture
# ============================================================
class FeatureCross(layers.Layer):
    """Compact cross layer for multiplicative feature interactions."""
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.proj = layers.Dense(units, use_bias=True)
        self.gate = layers.Dense(units, activation="sigmoid")

    def call(self, x):
        z = self.proj(x)
        g = self.gate(x)
        return z * g + x


class GatedResidualBlockV2(layers.Layer):
    def __init__(self, units, dropout_rate=0.12, l2_reg=2e-5, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        reg = keras.regularizers.l2(l2_reg)

        self.value = layers.Dense(units, activation="gelu", kernel_regularizer=reg)
        self.gate = layers.Dense(units, activation="sigmoid", kernel_regularizer=reg)
        self.mix = layers.Dense(units, activation="gelu", kernel_regularizer=reg)
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        x = self.norm1(inputs)
        v = self.value(x)
        g = self.gate(x)
        z = self.mix(v * g)
        z = self.dropout(z, training=training)
        return inputs + z


def build_residual_dgrnn(input_dim, width=192, blocks=4, dropout=0.12):
    inputs = keras.Input(shape=(input_dim,), name="dispatch_features")

    x = layers.Dense(width, activation="gelu")(inputs)
    x = layers.LayerNormalization()(x)

    # Feature-cross stage
    x = FeatureCross(width)(x)
    x = layers.Dropout(dropout)(x)

    for _ in range(blocks):
        x = GatedResidualBlockV2(width, dropout_rate=dropout)(x)

    x = layers.LayerNormalization()(x)
    x = layers.Dense(width // 2, activation="gelu")(x)
    x = layers.Dropout(dropout)(x)

    # Residual correction head. Prediction is added to RF baseline externally.
    residual_out = layers.Dense(1, activation="linear", name="residual")(x)
    class_out = layers.Dense(1, activation="sigmoid", name="high_cpu")(x)

    model = keras.Model(inputs, [residual_out, class_out], name="DGRNN_v2")
    return model


# ============================================================
# 15. Tail-aware residual loss
# ============================================================
# The weights are computed from TRAIN statistics only.
# ============================================================
residual_scale = float(np.std(residual_train) + 1e-6)
log_y_train = np.log1p(y_train.values.astype(np.float32))
log_q75 = float(np.quantile(log_y_train, 0.75))
log_median = float(np.median(log_y_train))
tail_width = max(log_q75 - log_median, 1e-3)


def tail_weighted_huber(y_true, y_pred):
    error = y_true - y_pred
    abs_error = tf.abs(error)
    delta = tf.constant(0.015, dtype=tf.float32)
    huber = tf.where(
        abs_error <= delta,
        0.5 * tf.square(error),
        delta * (abs_error - 0.5 * delta),
    )

    # y_true is the residual. The residual itself is not the CPU tail label,
    # therefore this loss uses a supplied training-time weight proxy via a
    # clipped residual magnitude. This keeps the residual learner stable.
    r = tf.abs(y_true) / tf.constant(residual_scale, dtype=tf.float32)
    w = 1.0 + 0.35 * tf.clip_by_value(r, 0.0, 2.0)
    return tf.reduce_mean(w * huber)


# ============================================================
# 16. Multi-task loss wrapper
# ============================================================
# Keras cannot directly use the original y for tail weighting because the
# regression head predicts residuals. We therefore use a stable weighted Huber
# on residuals + focal classification loss.
# ============================================================
def focal_bce(gamma=2.0, alpha=0.70):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        return tf.reduce_mean(-alpha_t * tf.pow(1 - pt, gamma) * tf.math.log(pt))
    return loss


# ============================================================
# 17. Train residual DGRNN
# ============================================================
print("\nTraining proposed DGRNN-v2 residual multi-task model...")

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_STATE + 100)

model = build_residual_dgrnn(
    input_dim=X_train_s.shape[1],
    width=192,
    blocks=4,
    dropout=0.12,
)

model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=2e-5,
        clipnorm=1.0,
    ),
    loss={
        "residual": tail_weighted_huber,
        "high_cpu": focal_bce(gamma=2.0, alpha=0.70),
    },
    loss_weights={
        "residual": 1.0,
        "high_cpu": 0.12,
    },
    metrics={
        "residual": [keras.metrics.MeanAbsoluteError(name="residual_mae")],
        "high_cpu": [keras.metrics.BinaryAccuracy(name="accuracy"),
                      keras.metrics.Precision(name="precision"),
                      keras.metrics.Recall(name="recall")],
    },
)

checkpoint_path = os.path.join(MODEL_DIR, "dgrnn_v2_best.keras")

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        min_delta=1e-6,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=2e-6,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=0,
    ),
]

# Validation residual target is defined from the FULL training RF model.
# This is not used to train the model; it is only for validation monitoring.
rf_val_residual = y_val.values.astype(np.float32) - rf_val.astype(np.float32)

history = model.fit(
    X_train_s,
    {
        "residual": residual_train,
        "high_cpu": y_train_c.values,
    },
    validation_data=(
        X_val_s,
        {
            "residual": rf_val_residual,
            "high_cpu": y_val_c.values,
        },
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

# ============================================================
# 18. Generate DGRNN residual predictions
# ============================================================
val_outputs = model.predict(X_val_s, verbose=0)
test_outputs = model.predict(X_test_s, verbose=0)

nn_val_residual = val_outputs[0].reshape(-1)
n_test_residual = test_outputs[0].reshape(-1)

# Residual correction is clipped to avoid unstable extrapolation.
res_clip = float(np.quantile(np.abs(residual_train), 0.995))
nn_val_residual = np.clip(nn_val_residual, -res_clip, res_clip)
nn_test_residual = np.clip(nn_test_residual, -res_clip, res_clip)

# Base RF + learned correction
residual_dgrnn_val = np.maximum(rf_val + nn_val_residual, 0)
residual_dgrnn_test = np.maximum(rf_test + nn_test_residual, 0)

# Also evaluate raw neural residual correction by itself
m = print_regression_result(
    "DGRNN-v2 Residual", y_test, residual_dgrnn_test
)
regression_results.append({"Model": "DGRNN-v2 Residual", **m})
predictions["DGRNN-v2 Residual"] = residual_dgrnn_test

# ============================================================
# 19. Validation-only blend: RF + DGRNN correction
# ============================================================
# Search a small grid. Weight is selected from VALIDATION only.
# ============================================================
best_alpha = None
best_val_rmse = np.inf

for alpha in np.arange(0.0, 1.01, 0.02):
    blend_val = np.maximum(
        (1.0 - alpha) * rf_val + alpha * residual_dgrnn_val,
        0,
    )
    rmse = np.sqrt(mean_squared_error(y_val, blend_val))
    if rmse < best_val_rmse:
        best_val_rmse = rmse
        best_alpha = float(alpha)

hybrid_test = np.maximum(
    (1.0 - best_alpha) * rf_test + best_alpha * residual_dgrnn_test,
    0,
)

print("\n" + "=" * 75)
print("VALIDATION-OPTIMIZED RF + DGRNN-v2 BLEND")
print("=" * 75)
print(f"DGRNN correction weight: {best_alpha:.2f}")
print(f"RF weight:               {1.0-best_alpha:.2f}")
print(f"Validation RMSE:         {best_val_rmse:.6f}")
print("=" * 75)

m = print_regression_result("Hybrid RF + DGRNN-v2", y_test, hybrid_test)
regression_results.append({"Model": "Hybrid RF + DGRNN-v2", **m})
predictions["Hybrid RF + DGRNN-v2"] = hybrid_test

# ============================================================
# 20. Final regression table
# ============================================================
regression_results_df = (
    pd.DataFrame(regression_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("\n" + "=" * 75)
print("FINAL REGRESSION RESULTS -- V2")
print("=" * 75)
print(regression_results_df.to_string(index=False))

regression_results_df.to_csv(
    os.path.join(TABLE_DIR, "regression_results_v2.csv"), index=False
)

# ============================================================
# 21. Actual-vs-predicted figure
# ============================================================
best_regression_name = regression_results_df.iloc[0]["Model"]
best_regression_pred = predictions[best_regression_name]

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, best_regression_pred, alpha=0.25, s=10)
lims = [
    min(float(y_test.min()), float(best_regression_pred.min())),
    max(float(y_test.max()), float(best_regression_pred.max())),
]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual CPU usage (Test set)")
ax.set_ylabel("Predicted CPU usage (Test set)")
ax.set_title(f"Best regression model: {best_regression_name}")
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "best_regression_actual_vs_predicted_v2.png"),
    dpi=300, bbox_inches="tight"
)
plt.show()

# ============================================================
# 22. Direct comparison: RF vs DGRNN-v2 vs Hybrid
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
compare_names = ["Random Forest", "DGRNN-v2 Residual", "Hybrid RF + DGRNN-v2"]
for ax, name in zip(axes, compare_names):
    pred = predictions[name]
    ax.scatter(y_test, pred, alpha=0.22, s=9)
    lo = min(float(y_test.min()), float(pred.min()))
    hi = max(float(y_test.max()), float(pred.max()))
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1.2)
    ax.set_title(name)
    ax.set_xlabel("Actual CPU")
    ax.set_ylabel("Predicted CPU")
    ax.grid(True)
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "rf_dgrnn_hybrid_comparison_v2.png"),
    dpi=300, bbox_inches="tight"
)
plt.show()

# ============================================================
# 23. Residual analysis
# ============================================================
residuals = y_test.values - best_regression_pred

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_regression_pred, residuals, alpha=0.25, s=10)
ax.axhline(0, linestyle="--")
ax.set_xlabel("Predicted CPU usage")
ax.set_ylabel("Residual (actual - predicted)")
ax.set_title(f"Residual analysis: {best_regression_name}")
ax.grid(True)
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "residual_analysis_v2.png"),
    dpi=300, bbox_inches="tight"
)
plt.show()

# ============================================================
# 24. Error by actual CPU quantile
# ============================================================
error_df = pd.DataFrame({
    "actual": y_test.values,
    "predicted": best_regression_pred,
})
error_df["abs_error"] = np.abs(error_df["actual"] - error_df["predicted"])
error_df["squared_error"] = (error_df["actual"] - error_df["predicted"]) ** 2
error_df["actual_quantile"] = pd.qcut(
    error_df["actual"], q=5, duplicates="drop"
)

quantile_report = (
    error_df.groupby("actual_quantile", observed=True)
    .agg(
        N=("actual", "size"),
        Mean_Actual=("actual", "mean"),
        Mean_Predicted=("predicted", "mean"),
        MAE=("abs_error", "mean"),
        RMSE=("squared_error", lambda s: float(np.sqrt(np.mean(s)))),
    )
    .reset_index()
)

print("\nERROR BY ACTUAL-CPU QUANTILE -- V2")
display(quantile_report)
quantile_report.to_csv(
    os.path.join(TABLE_DIR, "error_by_actual_cpu_quantile_v2.csv"), index=False
)

# ============================================================
# 25. RF feature importance
# ============================================================
feature_importance = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": rf.feature_importances_,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

print("\nRANDOM FOREST FEATURE IMPORTANCE -- V2")
display(feature_importance)
feature_importance.to_csv(
    os.path.join(TABLE_DIR, "random_forest_feature_importance_v2.csv"),
    index=False,
)

# ============================================================
# 26. Training curves
# ============================================================
def plot_history(history):
    hist = history.history
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(hist["loss"], label="Training loss")
    ax.plot(hist["val_loss"], label="Validation loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Joint loss")
    ax.set_title("DGRNN-v2 training history")
    ax.grid(True)
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        os.path.join(FIGURE_DIR, "dgrnn_v2_training_history.png"),
        dpi=300, bbox_inches="tight"
    )
    plt.show()

plot_history(history)

# ============================================================
# 27. Classification: Random Forest
# ============================================================
rf_c = RandomForestClassifier(
    n_estimators=350,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_c.fit(X_train, y_train_c.astype(int))

rf_val_prob = rf_c.predict_proba(X_val)[:, 1]
rf_test_prob = rf_c.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)

def best_f1_threshold(y_true, probs):
    best_t, best_f = 0.50, -1
    for t in thresholds:
        pred = (probs >= t).astype(int)
        f = f1_score(y_true, pred, zero_division=0)
        if f > best_f:
            best_f = f
            best_t = float(t)
    return best_t, best_f

rf_thr, rf_val_f1 = best_f1_threshold(y_val_c, rf_val_prob)
rf_test_class = (rf_test_prob >= rf_thr).astype(int)

# ============================================================
# 28. Classification: DGRNN-v2 auxiliary head
# ============================================================
nn_val_prob = val_outputs[1].reshape(-1)
nn_test_prob = test_outputs[1].reshape(-1)

nn_thr, nn_val_f1 = best_f1_threshold(y_val_c, nn_val_prob)
nn_test_class = (nn_test_prob >= nn_thr).astype(int)


def classification_metrics(name, y_true, y_pred):
    result = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }
    print(
        f"[{name}] Accuracy={result['Accuracy']:.6f} "
        f"Precision={result['Precision']:.6f} "
        f"Recall={result['Recall']:.6f} F1={result['F1']:.6f}"
    )
    return result

classification_results = [
    classification_metrics("Random Forest", y_test_c, rf_test_class),
    classification_metrics("DGRNN-v2 Multi-task", y_test_c, nn_test_class),
]

classification_results_df = (
    pd.DataFrame(classification_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

print("\nFINAL CLASSIFICATION RESULTS -- V2")
print(classification_results_df.to_string(index=False))
classification_results_df.to_csv(
    os.path.join(TABLE_DIR, "classification_results_v2.csv"), index=False
)

# ============================================================
# 29. Confusion matrix
# ============================================================
best_classifier_name = classification_results_df.iloc[0]["Model"]
best_class_pred = rf_test_class if best_classifier_name == "Random Forest" else nn_test_class

cm = confusion_matrix(y_test_c, best_class_pred)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not high-CPU", "High-CPU"],
).plot(ax=ax)
ax.set_title(f"Confusion matrix: {best_classifier_name}")
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "best_classifier_confusion_matrix_v2.png"),
    dpi=300, bbox_inches="tight"
)
plt.show()

# ============================================================
# 30. Classification threshold report
# ============================================================
threshold_report = pd.DataFrame({
    "Classifier": ["Random Forest", "DGRNN-v2 Multi-task"],
    "Validation_Selected_Threshold": [rf_thr, nn_thr],
    "Validation_F1": [rf_val_f1, nn_val_f1],
})
display(threshold_report)
threshold_report.to_csv(
    os.path.join(TABLE_DIR, "classification_thresholds_v2.csv"), index=False
)

# ============================================================
# 31. Save architecture / experiment configuration
# ============================================================
config = {
    "version": "DGRNN-v2",
    "random_state": RANDOM_STATE,
    "base_features": BASE_FEATURES,
    "engineered_features": list(X.columns),
    "target": TARGET,
    "raw_rows": int(len(df_raw)),
    "processed_rows": int(len(X)),
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_val)),
    "test_rows": int(len(X_test)),
    "high_cpu_threshold": high_cpu_threshold,
    "oof_folds": OOF_FOLDS,
    "rf_trees": RF_TREES,
    "extra_trees": ET_TREES,
    "dgrnn_width": 192,
    "dgrnn_blocks": 4,
    "dgrnn_dropout": 0.12,
    "batch_size": BATCH_SIZE,
    "epochs_max": EPOCHS,
    "early_stopping_patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "best_blend_alpha_dgrnn": best_alpha,
    "best_blend_alpha_rf": 1.0 - best_alpha,
    "classification_thresholds": {
        "Random Forest": rf_thr,
        "DGRNN-v2": nn_thr,
    },
    "post_execution_fields_excluded": POST_EXECUTION_FIELDS,
}

with open(os.path.join(TABLE_DIR, "experiment_configuration_v2.json"), "w") as f:
    json.dump(config, f, indent=2)

# ============================================================
# 32. Save model summary
# ============================================================
with open(os.path.join(TABLE_DIR, "dgrnn_v2_model_summary.txt"), "w") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

# ============================================================
# 33. Paper-ready summary
# ============================================================
best_reg = regression_results_df.iloc[0]
best_clf = classification_results_df.iloc[0]

print("\n" + "=" * 80)
print("FINAL PAPER-READY SUMMARY -- DGRNN-v2")
print("=" * 80)
print(f"Raw rows:                  {len(df_raw):,}")
print(f"Rows after preprocessing:  {len(X):,}")
print(f"Train / Validation / Test: {len(X_train):,} / {len(X_val):,} / {len(X_test):,}")
print(f"High-CPU threshold:        {high_cpu_threshold:.6f}")
print("\nBEST REGRESSION MODEL")
print(f"Model: {best_reg['Model']}")
print(f"MAE:   {best_reg['MAE']:.6f}")
print(f"RMSE:  {best_reg['RMSE']:.6f}")
print(f"R2:    {best_reg['R2']:.6f}")
print("\nBEST CLASSIFICATION MODEL")
print(f"Model:      {best_clf['Model']}")
print(f"Accuracy:   {best_clf['Accuracy']:.6f}")
print(f"Precision:  {best_clf['Precision']:.6f}")
print(f"Recall:     {best_clf['Recall']:.6f}")
print(f"F1:         {best_clf['F1']:.6f}")
print("\nVALIDATION BLEND")
print(f"RF weight:      {1-best_alpha:.2f}")
print(f"DGRNN weight:   {best_alpha:.2f}")
print("\nResults saved to:")
print(RESULTS_DIR)
print("Tables:", TABLE_DIR)
print("Figures:", FIGURE_DIR)
print("Models:", MODEL_DIR)
print("=" * 80)

print("\nMethodological safeguards:")
print("- Dispatch-time features only")
print("- Resource requests parsed from raw nested fields")
print("- Scaling fitted on TRAIN only")
print("- RF residuals learned from 3-fold OOF predictions")
print("- High-CPU threshold computed from TRAIN only")
print("- Blend weight selected from VALIDATION only")
print("- Early stopping restores best validation checkpoint")
print("- TEST set reserved for final evaluation")

Mounted at /content/drive
TensorFlow: 2.20.0
Results: /content/drive/MyDrive/AI-GoogleCluster/results_enhanced_dgrnn_v2
Loaded cached dataset: /content/drive/MyDrive/AI-GoogleCluster/borg_traces_data.csv
Shape: (405894, 34)

Columns:
['Unnamed: 0', 'time', 'instance_events_type', 'collection_id', 'scheduling_class', 'collection_type', 'priority', 'alloc_collection_id', 'instance_index', 'machine_id', 'resource_request', 'constraint', 'collections_events_type', 'user', 'collection_name', 'collection_logical_name', 'start_after_collection_ids', 'vertical_scaling', 'scheduler', 'start_time', 'end_time', 'average_usage', 'maximum_usage', 'random_sample_usage', 'assigned_memory', 'page_cache_memory', 'cycles_per_instruction', 'memory_accesses_per_instruction', 'sample_rate', 'cpu_usage_distribution', 'tail_cpu_usage_distribution', 'cluster', 'event', 'failed']

Raw rows: 405,894
Missing parsed values:
resource_request_cpus      774
resource_request_memory    774
realized_cpu_usage          